# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manahilrubabsatti/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [3]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face connection ready.")

Hugging Face connection ready.


In [7]:
import pandas as pd
import numpy as np
import os

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

I will rank content using a simple action score based on observed search performance in March 2026.

Pages with meaningful search impressions but weak CTR will receive a higher priority because they have visibility but are not generating many clicks. Pages with weaker search positions will also receive higher priority when they still have meaningful impressions, because improving visibility may be an opportunity for review.

The score is a prioritisation rule, not a prediction of future performance.

Reason codes:

- **LOW_CTR** — the page has meaningful search impressions but relatively low CTR.
- **WEAK_POSITION** — the page has meaningful search impressions but a weak average search position.
- **LOW_VISIBILITY** — the page has very little search visibility, so the opportunity is uncertain.
- **REVIEW** — the page does not strongly match the main signals but is still included in the ranked queue.

Action:

- **REFRESH** — prioritise the page for content review.
- **MONITOR** — keep the page for observation rather than immediate refresh.

In [4]:
# ML-07 — Load March 2026 development data

baseline = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
HAVING SUM(gsc_impressions) > 0
""").df()

baseline["ctr"] = (
    baseline["clicks"].astype(float)
    / baseline["impressions"].astype(float)
)

print("Rows:", len(baseline))
display(baseline.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 176738


,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,6.893301,0.001073
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,3.214128,0.000000
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.535346,0.001066
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.435680,0.002629
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,3.871795,0.002331


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 2. Build the ranked queue

I use two main signals in the baseline score: CTR and average search position.

Low CTR among pages that already receive impressions can indicate an opportunity for review. A weaker search position is also treated as a directional opportunity signal.

I use the measured March 2026 distribution to create simple percentile-based components. This avoids choosing arbitrary absolute thresholds and keeps the rule tied to the observed data.

The final score is a prioritisation score only. A higher score means higher review priority; it does not mean that a refresh will definitely improve performance.

In [12]:
# ML-07 — Build baseline action score

# Rank-based components:
# Higher = greater potential review priority

baseline["ctr_rank"] = baseline["ctr"].rank(pct=True)
baseline["position_rank"] = baseline["avg_position"].rank(pct=True)

# Low CTR = higher priority
baseline["low_ctr_score"] = 1 - baseline["ctr_rank"]

# Worse position = higher priority
baseline["weak_position_score"] = baseline["position_rank"]

# Small weight for visibility so pages with almost no impressions
# do not dominate the queue.
baseline["visibility_score"] = (
    baseline["impressions"].rank(pct=True)
)

baseline["action_score"] = (
    0.50 * baseline["low_ctr_score"]
    + 0.35 * baseline["weak_position_score"]
    + 0.15 * baseline["visibility_score"]
)

# Reason code based on the strongest observed signal(s)

low_ctr = (
    (baseline["ctr"] <= baseline["ctr"].quantile(0.25)) &
    (baseline["impressions"] >= baseline["impressions"].median())
)

weak_position = (
    (baseline["avg_position"] >= baseline["avg_position"].quantile(0.75)) &
    (baseline["impressions"] >= baseline["impressions"].median())
)

baseline["reason_code"] = np.select(
    [
        low_ctr & weak_position,
        low_ctr,
        weak_position,
    ],
    [
        "LOW_CTR_AND_WEAK_POSITION",
        "LOW_CTR",
        "WEAK_POSITION",
    ],
    default="REVIEW"
)

# Action
baseline["action"] = np.where(
    baseline["action_score"] >= baseline["action_score"].quantile(0.75),
    "REFRESH",
    "MONITOR"
)

# Rank
baseline = baseline.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

baseline["rank"] = np.arange(1, len(baseline) + 1)

# Confidence note
baseline["confidence_note"] = np.select(
    [
        baseline["impressions"] >= baseline["impressions"].quantile(0.75),
        baseline["impressions"] >= baseline["impressions"].median(),
    ],
    [
        "Higher visibility; stronger evidence for review priority.",
        "Moderate visibility; directional evidence.",
    ],
    default="Low visibility; treat as lower-confidence opportunity."
)

# Final output columns
baseline_output = baseline[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "action_score",
        "impressions",
        "clicks",
        "ctr",
        "avg_position",
    ]
].copy()

# Write CSV
import os

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

baseline_output.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Total ranked rows:", len(baseline_output))

display(baseline_output.head(20))

Saved: work/outputs/baseline_action_score.csv
Total ranked rows: 176738


,rank,client_hash_id,content_hash_id,action,reason_code,confidence_note,action_score,impressions,clicks,ctr,avg_position
0,1,client_3197e6291363b4db,content_65b8a4998e633d89,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,0.836669,8905.0,0.0,0.0,76.919708
1,2,client_23a62021009f63c4,content_295e883e0e86ca3c,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,0.830764,21939.0,0.0,0.0,61.379142
2,3,client_3197e6291363b4db,content_2392ac0360e7c84c,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,0.823737,3309.0,0.0,0.0,74.363856
3,4,client_ff644d8251367cbb,content_c8532cca37ba990c,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,0.823512,4690.0,0.0,0.0,66.943070
4,5,client_e547b89c05043229,content_4002467a580a7f98,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,0.823311,11973.0,0.0,0.0,54.937443
5,6,client_3197e6291363b4db,content_b49acf92cc1c8c7e,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,0.823074,2478.0,0.0,0.0,83.020985
6,7,client_e547b89c05043229,content_a7511e24c1a02cf2,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,0.821782,2693.0,0.0,0.0,76.427033
7,8,client_e547b89c05043229,content_4f4acdb0a3d60255,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,0.821003,3727.0,0.0,0.0,67.597800
8,9,client_23a62021009f63c4,content_959d535a9fcc865c,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,0.820917,10452.0,0.0,0.0,53.172407
9,10,client_e547b89c05043229,content_d07ea12b5ba91c34,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,0.820210,2385.0,0.0,0.0,77.131656


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 review

The top 20 rows are the highest-priority items produced by the baseline action score.

All 20 have observed search impressions but zero clicks, giving them a CTR of zero. They also have relatively weak average search positions. Therefore, the baseline identifies them as directional refresh opportunities based on both low CTR and weak position.

Confidence is stronger for pages with more impressions because the signals are based on more observed search activity. However, these recommendations can still be wrong because CTR can be affected by search intent, SERP features, branding, and other factors not represented in the baseline.

The queue is therefore a decision-support list for human review, not a claim that every selected page should definitely be refreshed.

In [13]:
# ML-07 — Top-20 review

top20 = baseline_output.head(20).copy()

top20["what_would_make_it_wrong"] = (
    "Search intent, SERP features, or other context may explain "
    "the low CTR or weak position; the score does not prove "
    "that a refresh will improve performance."
)

display(
    top20[
        [
            "rank",
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong",
        ]
    ]
)

,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,"Search intent, SERP features, or other context..."
1,2,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,"Search intent, SERP features, or other context..."
2,3,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,"Search intent, SERP features, or other context..."
3,4,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,"Search intent, SERP features, or other context..."
4,5,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,"Search intent, SERP features, or other context..."
5,6,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,"Search intent, SERP features, or other context..."
6,7,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,"Search intent, SERP features, or other context..."
7,8,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,"Search intent, SERP features, or other context..."
8,9,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,"Search intent, SERP features, or other context..."
9,10,REFRESH,LOW_CTR_AND_WEAK_POSITION,Higher visibility; stronger evidence for revie...,"Search intent, SERP features, or other context..."


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

Some highly ranked items may still be weak picks. A page can have zero CTR because of search intent, SERP features, branding, or other factors rather than because its content needs a refresh. A weak average position can also have causes that are not captured by this baseline.

The lowest-visibility items in the top 20 should therefore be treated with more caution than pages with much higher impressions.

For leakage, the baseline uses only March 2026 observed search-performance data: impressions, clicks, CTR, and average position. It does not use future performance, product flags, client names, URLs, or private search queries.

In [14]:
# ML-07 — Weak picks + leakage check

print("Top-20 action counts:")
print(top20["action"].value_counts())

print("\nTop-20 reason codes:")
print(top20["reason_code"].value_counts())

print("\nLowest-visibility items among Top 20:")
display(
    top20.nsmallest(5, "impressions")[
        [
            "rank",
            "action",
            "reason_code",
            "impressions",
            "clicks",
            "ctr",
            "avg_position",
            "action_score",
        ]
    ]
)

# Leakage / privacy checks
print("\nLeakage checks:")

print("Development window: March 2026")
print("Future-window performance used: NO")
print("Product flags used: NO")
print("Client names used: NO")
print("URLs used: NO")
print("Private search queries used: NO")

Top-20 action counts:
action
REFRESH    20
Name: count, dtype: int64

Top-20 reason codes:
reason_code
LOW_CTR_AND_WEAK_POSITION    20
Name: count, dtype: int64

Lowest-visibility items among Top 20:


,rank,action,reason_code,impressions,clicks,ctr,avg_position,action_score
10,11,REFRESH,LOW_CTR_AND_WEAK_POSITION,2186.0,0.0,0.0,79.677036,0.819856
9,10,REFRESH,LOW_CTR_AND_WEAK_POSITION,2385.0,0.0,0.0,77.131656,0.820210
5,6,REFRESH,LOW_CTR_AND_WEAK_POSITION,2478.0,0.0,0.0,83.020985,0.823074
15,16,REFRESH,LOW_CTR_AND_WEAK_POSITION,2636.0,0.0,0.0,70.834598,0.818212
6,7,REFRESH,LOW_CTR_AND_WEAK_POSITION,2693.0,0.0,0.0,76.427033,0.821782



Leakage checks:
Development window: March 2026
Future-window performance used: NO
Product flags used: NO
Client names used: NO
URLs used: NO
Private search queries used: NO


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.